In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

df = pd.read_csv('KO_weekly_return_volatility.csv')
daily_data = pd.read_csv('KO_weekly_return_detailed.csv')

train = df[df['Year'].isin([2021, 2022, 2023])].reset_index(drop=True)
test = df[df['Year'].isin([2024, 2025])].reset_index(drop=True)
daily_test = daily_data[daily_data['Year'].isin([2024, 2025])].reset_index(drop=True)

X_train = train[['mean_return', 'volatility']].values
Y_train = train['decision'].values
X_test = test[['mean_return', 'volatility']].values
Y_test = test['decision'].values

print(X_train.shape)
# entropy tree
dt_entropy = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_entropy.fit(X_train, Y_train)
preds_entropy = dt_entropy.predict(X_test)

# gini tree
dt_gini = DecisionTreeClassifier(criterion='gini', random_state=42)
dt_gini.fit(X_train, Y_train)
preds_gini = dt_gini.predict(X_test)



(156, 2)


## Q1 : Implement decision tree and compute accuracy


In [9]:
# Q1: accuracy
entropy_acc = round(dt_entropy.score(X_test, Y_test) * 100, 2)
print(f'Entropy accuracy: {entropy_acc}%')

Entropy accuracy: 51.89%


## Q2 : Confusion matrix


In [16]:
# Q2: confusion matrix
cm = confusion_matrix(Y_test, preds_entropy, labels=['YES', 'NO'])
print('Confusion Matrix (Entropy):')
print(f'  Predicted YES  Predicted NO')
print(f'Actual YES  {cm[0][0]}           {cm[0][1]}')
print(f'Actual NO   {cm[1][0]}           {cm[1][1]}')

Confusion Matrix (Entropy):
  Predicted YES  Predicted NO
Actual YES  12           21
Actual NO   30           43


## Q3 : True positive rate (sensitivity)


In [17]:
# Q3: TPR
TP, FN, FP, TN = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
print(f'TPR (sensitivity): {round(TP / (TP + FN) * 100, 2)}%')

TPR (sensitivity): 36.36%


## Q4 : True negative rate (specificity)


In [26]:
# Q4: TNR
print(f'TNR (specificity): {round(TN / (TN + FP) * 100, 2)}%')

TNR (specificity): 58.9%


## Q5 : Trading strategy vs buy-and-hold


In [30]:
# Q5: trading strategy (entropy)
test['entropy_prediction'] = preds_entropy

initial_balance = 100
entropy_balance = initial_balance
position = None

for week_idx in range(len(test)):
    current_week = test.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']
    week_data = daily_test[(daily_test['Year'] == year) & (daily_test['Week_Number'] == week_num)]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                entropy_balance = entropy_balance * (1 + row['Return'] / 100)

    if week_idx < len(test) - 1:
        next_prediction = test.iloc[week_idx + 1]['entropy_prediction']
        position = 'invested' if next_prediction == 'YES' else None

bh_balance = initial_balance
for idx, row in daily_test.iterrows():
    if pd.notna(row['Return']):
        bh_balance = bh_balance * (1 + row['Return'] / 100)

print(f'Entropy Strategy Final Balance: ${entropy_balance:.2f}')
print(f'Buy and Hold Final Balance: ${bh_balance:.2f}')
winner = 'Entropy Strategy' if entropy_balance > bh_balance else 'Buy-and-Hold'
print(f'{winner} results in a larger amount at the end of the testing period')

Entropy Strategy Final Balance: $133.90
Buy and Hold Final Balance: $126.14
Entropy Strategy results in a larger amount at the end of the testing period


## Q6 : Same with Gini criteria


In [34]:
# Q6: gini criteria
cm_gini = confusion_matrix(Y_test, preds_gini, labels=['YES', 'NO'])
gini_acc = round(dt_gini.score(X_test, Y_test) * 100, 2)
TP_g, FN_g, FP_g, TN_g = cm_gini[0][0], cm_gini[0][1], cm_gini[1][0], cm_gini[1][1]

print(f'\nGini accuracy: {gini_acc}%')
print('Confusion Matrix (Gini):')
print(f'  Predicted YES  Predicted NO')
print(f'Actual YES  {cm_gini[0][0]}           {cm_gini[0][1]}')
print(f'Actual NO   {cm_gini[1][0]}           {cm_gini[1][1]}')
print(f'TPR (sensitivity): {round(TP_g / (TP_g + FN_g) * 100, 2)}%')
print(f'TNR (specificity): {round(TN_g / (TN_g + FP_g) * 100, 2)}%')

test['gini_prediction'] = preds_gini
gini_balance = initial_balance
position = None

for week_idx in range(len(test)):
    current_week = test.iloc[week_idx]
    year = current_week['Year']
    week_num = current_week['Week_Number']
    week_data = daily_test[(daily_test['Year'] == year) & (daily_test['Week_Number'] == week_num)]

    if len(week_data) == 0:
        continue

    if position == 'invested':
        for idx, row in week_data.iterrows():
            if pd.notna(row['Return']):
                gini_balance = gini_balance * (1 + row['Return'] / 100)

    if week_idx < len(test) - 1:
        next_prediction = test.iloc[week_idx + 1]['gini_prediction']
        position = 'invested' if next_prediction == 'YES' else None

print(f'\nGini Strategy Final Balance: ${gini_balance:.2f}')
print(f'Buy and Hold Final Balance: ${bh_balance:.2f}')
winner_gini = 'Gini Strategy' if gini_balance > bh_balance else 'Buy-and-Hold'
print(f'{winner_gini} results in a larger amount at the end of the testing period')



Gini accuracy: 57.55%
Confusion Matrix (Gini):
  Predicted YES  Predicted NO
Actual YES  12           21
Actual NO   24           49
TPR (sensitivity): 36.36%
TNR (specificity): 67.12%

Gini Strategy Final Balance: $131.72
Buy and Hold Final Balance: $126.14
Gini Strategy results in a larger amount at the end of the testing period


## Q7 : Which criteria is better (accuracy, TPR, TNR)


In [33]:
# Q7: which criteria is better
print(f'\nComparison:')
print(f'Accuracy  - Entropy: {entropy_acc}%, Gini: {gini_acc}%')
print(f'TPR       - Entropy: {round(TP/(TP+FN)*100,2)}%, Gini: {round(TP_g/(TP_g+FN_g)*100,2)}%')
print(f'TNR       - Entropy: {round(TN/(TN+FP)*100,2)}%, Gini: {round(TN_g/(TN_g+FP_g)*100,2)}%')
better = 'Entropy' if entropy_acc >= gini_acc else 'Gini'
print(f'{better} criterion is better based on accuracy')


Comparison:
Accuracy  - Entropy: 51.89%, Gini: 57.55%
TPR       - Entropy: 36.36%, Gini: 36.36%
TNR       - Entropy: 58.9%, Gini: 67.12%
Gini criterion is better based on accuracy
